# ATLAS | WRI 13-score validation, EDA & optional S3 publication

**Final replacement for the legacy 17-score merge notebook.** This notebook consumes the **already completed** spatial join outputs; it does not rerun geographic matching or modify the original WRI polygons. It validates the original facility data, the 13-score raw merge, and the 0–1 normalized merge; creates a WRI-only feature table; evaluates the five flagged indicators; and provides an **opt-in** S3 publication stage.

**Research dataset:** retain all 13 standard Aqueduct indicators. **Proposed baseline:** nine features; this is a modeling decision, not a change to the published research dataset. No ELP scores are used.

Run from your ATLAS repository root. Update paths in the configuration cell if necessary. S3 upload is disabled by default.

## 1 — Configuration and imports

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path('Data/processed')
if not (DATA_DIR / 'im3_eia_combined.csv').exists():
    DATA_DIR = Path('/mnt/data') if Path('/mnt/data/im3_eia_combined.csv').exists() else DATA_DIR
ORIGINAL_PATH = DATA_DIR / 'im3_eia_combined.csv'
RAW_PATH = DATA_DIR / 'im3_eia_wri_13scores_raw.csv'
NORM_PATH = DATA_DIR / 'im3_eia_wri_13scores_normalized_0to1.csv'
FEATURE_PATH = DATA_DIR / 'wri_aqueduct_facility_features_13scores.csv'

SCORES = ['bws_score','bwd_score','iav_score','sev_score','gtd_score',
          'rfr_score','cfr_score','drr_score','ucw_score','cep_score',
          'udw_score','usa_score','rri_score']
BASELINE_9 = ['bws_score','bwd_score','iav_score','sev_score',
              'rfr_score','cfr_score','drr_score','cep_score','udw_score']
ELP = ['w_awr_elp_qan_score','w_awr_elp_qal_score',
       'w_awr_elp_rrr_score','w_awr_elp_tot_score']
assert len(SCORES) == 13 and len(BASELINE_9) == 9
print('Data directory:', DATA_DIR.resolve())

Data directory: /mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS/Data/processed


## temporary check on 2 merge versions

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("Data/processed")

old = pd.read_csv(
    DATA_DIR / "im3_eia_combined.csv",
    dtype={"id": str}
)

new = pd.read_csv(
    DATA_DIR / "im3_atlas_combined.csv",
    dtype={"id": str}
)

print("=== DATASET SHAPES ===")
print("IM3/EIA:", old.shape)
print("ATLAS:", new.shape)

print("\n=== COLUMN DIFFERENCES ===")
print("Only in IM3/EIA:", sorted(set(old.columns) - set(new.columns)))
print("Only in ATLAS:", sorted(set(new.columns) - set(old.columns)))

print("\n=== FACILITY ID DIFFERENCES ===")
print("Only in IM3/EIA:", len(set(old.id) - set(new.id)))
print("Only in ATLAS:", len(set(new.id) - set(old.id)))

print("\n=== DUPLICATE IDS ===")
print("IM3/EIA:", old.id.duplicated().sum())
print("ATLAS:", new.id.duplicated().sum())

# Compare values using facility ID rather than row position.
assert old.id.is_unique and new.id.is_unique

common_ids = old.set_index("id").index.intersection(
    new.set_index("id").index
)

common_cols = sorted(
    (set(old.columns) & set(new.columns)) - {"id"}
)

a = old.set_index("id").loc[common_ids, common_cols]
b = new.set_index("id").loc[common_ids, common_cols]

# Ignore insignificant floating-point differences.
changes = {}

for col in common_cols:
    x, y = a[col], b[col]

    if pd.api.types.is_numeric_dtype(x) and pd.api.types.is_numeric_dtype(y):
        same = np.isclose(
            x.to_numpy(dtype=float),
            y.to_numpy(dtype=float),
            rtol=1e-9,
            atol=1e-10,
            equal_nan=True
        )
    else:
        same = x.astype("string").fillna("<MISSING>").eq(
            y.astype("string").fillna("<MISSING>")
        ).to_numpy()

    count = int((~same).sum())

    if count:
        changes[col] = count

print("\n=== CHANGED VALUES BY COLUMN ===")
print(pd.Series(changes).sort_values(ascending=False)
      if changes else "No differences in shared columns")

=== DATASET SHAPES ===
IM3/EIA: (1147, 50)
ATLAS: (1147, 49)

=== COLUMN DIFFERENCES ===
Only in IM3/EIA: ['est_hourly_energy_cost_usd', 'est_mwh_per_hour', 'period']
Only in ATLAS: ['county_fips', 'retail_period']

=== FACILITY ID DIFFERENCES ===
Only in IM3/EIA: 0
Only in ATLAS: 0

=== DUPLICATE IDS ===
IM3/EIA: 0
ATLAS: 0

=== CHANGED VALUES BY COLUMN ===
comm_elec_price_cents_kwh    1147
comm_elec_sales_gwh          1147
dtype: int64


## 2 — Read CSVs, preserving facility identifiers

In [ ]:
for p in (ORIGINAL_PATH, RAW_PATH, NORM_PATH):
    assert p.exists(), f'Missing input: {p}'
original = pd.read_csv(ORIGINAL_PATH, dtype={'id': 'string'})
raw = pd.read_csv(RAW_PATH, dtype={'id': 'string', 'pfaf_id': 'string'})
normalized = pd.read_csv(NORM_PATH, dtype={'id': 'string', 'pfaf_id': 'string'})
for name, df in [('original', original), ('raw', raw), ('normalized', normalized)]:
    assert 'id' in df.columns and df['id'].notna().all() and df['id'].is_unique, name
    print(f'{name}: {df.shape}, unique IDs={df.id.nunique()}')

## 3 — Schema, facility preservation, and score checks

In [ ]:
assert set(original.id) == set(raw.id) == set(normalized.id), 'Facility ID sets differ'
assert set(original.columns).issubset(raw.columns)
assert set(original.columns).issubset(normalized.columns)
assert set(SCORES).issubset(raw.columns)
NORM_COLS = [s.removesuffix('_score') + '_norm' for s in SCORES]
assert set(NORM_COLS).issubset(normalized.columns)
assert not any(s in normalized.columns for s in SCORES)
assert not (set(ELP) & (set(raw.columns) | set(normalized.columns)))
assert 'pfaf_id' in raw.columns and 'pfaf_id' in normalized.columns
assert raw.pfaf_id.notna().all() and normalized.pfaf_id.notna().all()
assert raw.shape[1] == original.shape[1] + 14
assert normalized.shape[1] == original.shape[1] + 14

orig_idx = original.set_index('id').sort_index()
raw_idx = raw.set_index('id').sort_index()
norm_idx = normalized.set_index('id').sort_index()
for name, target in [('raw',raw_idx),('normalized',norm_idx)]:
    for col in orig_idx.columns:
        a, b = orig_idx[col], target[col]
        if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
            assert np.allclose(a.to_numpy(dtype=float), b.to_numpy(dtype=float), rtol=1e-10, atol=1e-10, equal_nan=True), (name,col)
        else:
            assert a.astype('string').fillna('<NA>').equals(b.astype('string').fillna('<NA>')), (name,col)
assert raw_idx['pfaf_id'].equals(norm_idx['pfaf_id'])
for s in SCORES:
    x = pd.to_numeric(raw_idx[s], errors='raise')
    y = pd.to_numeric(norm_idx[s.removesuffix('_score') + '_norm'], errors='raise')
    assert x.dropna().between(0,5).all(), s
    assert x.isna().equals(y.isna()), f'Missingness mismatch: {s}'
    assert np.allclose(x.to_numpy(), (y*5).to_numpy(), rtol=1e-10, atol=1e-10, equal_nan=True), s
    assert y.dropna().between(0,1).all(), s
print('PASS: identifiers, 50 source fields, 13 scores, ELP exclusion, watershed IDs, score ranges, missingness, and fixed scaling.')

## 4 — Facility-level EDA and feature-quality report

In [ ]:
score_data = raw[SCORES].apply(pd.to_numeric, errors='raise')
report = pd.DataFrame(index=SCORES)
report['missing_n'] = score_data.isna().sum()
report['missing_pct'] = (100 * score_data.isna().mean()).round(2)
report['unique_nonmissing'] = score_data.nunique(dropna=True)
report['zero_n'] = score_data.eq(0).sum()
report['nonzero_n'] = score_data.notna().sum() - report['zero_n']
report['zero_pct_observed'] = (100 * report['zero_n'] / score_data.notna().sum()).round(2)
report['min'] = score_data.min()
report['max'] = score_data.max()
display(report)
print('Distinct matched watershed IDs:', raw['pfaf_id'].nunique())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
report['missing_pct'].sort_values().plot.barh(ax=axes[0], title='WRI missingness (%)')
axes[0].set_xlabel('% missing')
report['unique_nonmissing'].sort_values().plot.barh(ax=axes[1], title='Distinct observed values')
axes[1].set_xlabel('Unique values')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, s in zip(axes.flat, BASELINE_9):
    score_data[s].dropna().hist(ax=ax, bins=25)
    ax.set_title(s)
    ax.set_xlabel('Raw WRI score (0–5)')
plt.tight_layout()
plt.show()

In [ ]:
corr = score_data[BASELINE_9].corr()
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr.to_numpy(), vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(BASELINE_9)), BASELINE_9, rotation=60, ha='right')
ax.set_yticks(range(len(BASELINE_9)), BASELINE_9)
fig.colorbar(im, ax=ax, label='Pearson correlation')
ax.set_title('Baseline WRI score correlations (facility-level)')
plt.tight_layout()
plt.show()

## 5 — Five indicators requiring attention

- **GTD (`gtd_score`)**: high missingness; exclude from the initial baseline and test a separate train-only imputation + missingness-flag experiment.
- **UCW (`ucw_score`)** and **RRI (`rri_score`)**: constant in this facility sample; exclude from model inputs.
- **USA (`usa_score`)**: almost entirely zero; exclude from initial baseline, reconsider with ablation.
- **CFR (`cfr_score`)**: many legitimate zeros; **retain** in the initial baseline.

These are **proposed** modeling choices; verify them on the actual training population (including negative examples), not just the positive-facility inventory. Do not remove any of the 13 scores from the archival research dataset.

In [ ]:
flagged = report.loc[['gtd_score','ucw_score','rri_score','usa_score','cfr_score']]
display(flagged)
assert set(BASELINE_9).issubset(SCORES)
print('Proposed baseline:', BASELINE_9)
print('Excluded from first baseline:', sorted(set(SCORES) - set(BASELINE_9)))

## 6 — Create and validate the WRI-only 13-score feature table

Only the **WRI-only** table is newly generated. Existing raw and normalized combined datasets are treated as validated inputs and are never overwritten.

In [ ]:
feature_table = raw[['id','pfaf_id'] + SCORES].copy()
assert feature_table.shape == (len(raw), 15)
assert feature_table['id'].is_unique
if FEATURE_PATH.exists():
    existing = pd.read_csv(FEATURE_PATH, dtype={'id':'string','pfaf_id':'string'})
    pd.testing.assert_frame_equal(
        existing.sort_values('id').reset_index(drop=True),
        feature_table.sort_values('id').reset_index(drop=True),
        check_dtype=False, check_exact=False, rtol=1e-10, atol=1e-10)
    print('Existing WRI-only CSV verified:', FEATURE_PATH)
else:
    FEATURE_PATH.parent.mkdir(parents=True, exist_ok=True)
    feature_table.to_csv(FEATURE_PATH, index=False)
    saved = pd.read_csv(FEATURE_PATH, dtype={'id':'string','pfaf_id':'string'})
    assert len(saved) == len(raw) and saved.id.is_unique
    print('Created WRI-only CSV:', FEATURE_PATH)

## 7 — Optional S3 publication (disabled by default)

Set `PUBLISH_TO_S3=True` **only after** confirming the bucket and keys with your team. The notebook refuses to overwrite existing S3 objects. It uploads three validated files to distinct, versioned keys and verifies their SHA-256 hashes by downloading them back. AWS credentials must already be configured in your environment.

**No S3 changes happen with the default configuration.**

In [ ]:
PUBLISH_TO_S3 = False
S3_BUCKET = 'atlas-acm-research-f26'  # verify with your team
S3_PREFIX = 'dataset/wri_13score_v1'  # new versioned prefix; verify naming convention
files_to_publish = {
    'im3_eia_wri_13scores_raw.csv': RAW_PATH,
    'im3_eia_wri_13scores_normalized_0to1.csv': NORM_PATH,
    'wri_aqueduct_facility_features_13scores.csv': FEATURE_PATH,
}
print('S3 publication enabled:', PUBLISH_TO_S3)
for name, path in files_to_publish.items():
    print(f's3://{S3_BUCKET}/{S3_PREFIX}/{name} <- {path}')

In [ ]:
if PUBLISH_TO_S3:
    import boto3, hashlib
    from botocore.exceptions import ClientError
    s3 = boto3.client('s3')
    # Preflight ALL keys before uploading any object.
    for name in files_to_publish:
        key = f'{S3_PREFIX.rstrip("/")}/{name}'
        try:
            s3.head_object(Bucket=S3_BUCKET, Key=key)
        except ClientError as exc:
            status = exc.response.get('ResponseMetadata', {}).get('HTTPStatusCode')
            if status != 404:
                raise
        else:
            raise FileExistsError(f'Refusing overwrite: s3://{S3_BUCKET}/{key}')
    for name, local_path in files_to_publish.items():
        key = f'{S3_PREFIX.rstrip("/")}/{name}'
        data = local_path.read_bytes()
        # Conditional create guards against an object appearing after preflight.
        s3.put_object(Bucket=S3_BUCKET, Key=key, Body=data, ContentType='text/csv', IfNoneMatch='*')
        returned = s3.get_object(Bucket=S3_BUCKET, Key=key)['Body'].read()
        assert hashlib.sha256(data).digest() == hashlib.sha256(returned).digest()
        print('Uploaded and verified:', f's3://{S3_BUCKET}/{key}')
else:
    print('Skipped S3 publication (safe default).')

## 8 — Handoff to multimodal training

- Keep `im3_eia_wri_13scores_raw.csv` as the authoritative, unmodified facility-level research dataset.
- Keep the normalized 0–1 version for fixed-scale model inputs; it is **not** an imputed dataset.
- Use `BASELINE_9` for the first WRI ablation; do not confuse this feature subset with the 13-score archival schema.
- When adding non-facility negative samples, repeat the same spatial join and validate the resulting feature distributions before training.
- Fit any learned imputation, standardization, or feature selection **on training data only**. Evaluate random versus geographically grouped holdouts to assess spatial shortcut learning.
- A successful spatial join confirms containment, not correctness of source facility coordinates; perform geographic QA separately.